<a href="https://colab.research.google.com/github/Emanuel-Izaguirre-Cortes-03/PROGRAMACION-DE-SISTEMAS-DE-BASE-I---Grupo-K/blob/main/Izaguirre%20Emanuel%20%20PSSB%20I%20analizador_lexico_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GUÍA PRÁCTICA Y EVALUACIÓN ASÍNCRONA: ANALIZADORES LÉXICOS EN PYTHON Y JAVA
**Tema: Generación Automática de Analizadores Léxicos**  
**Asignatura:** Programación de Sistemas de Bas 1  

---  
## 🎯 Objetivos de Aprendizaje
1. **Comprender** la generación automática de analizadores léxicos a partir de especificaciones formales y expresiones regulares.
2. **Implementar y extender** escáneres léxicos en dos entornos tecnológicos representativos: **PLY (`ply.lex`)** en Python y **JFlex** en Java.
3. **Analizar** los mecanismos de disambiguación léxica (palabras reservadas vs. identificadores, coincidencia más larga) y el manejo de atributos/posiciones.
4. **Evaluar** estructuralmente las diferencias entre enfoques interpretados/dinámicos (PLY) y compilados/estáticos (JFlex).

---  
## 🐍 PARTE 1: Analizador Léxico en Python utilizando PLY (`ply.lex`)

### 1.1 Código Base e Instalación

In [1]:
# Instalación de la librería PLY en el entorno de Google Colab
!pip install ply > /dev/null

In [2]:
%%writefile lexer_ply.py

import ply.lex as lex

class PythonLexerPLY:
    """Especificación e implementación de un Analizador Léxico en Python usando PLY."""

    # 1. LISTA OBLIGATORIA DE TOKENS
    tokens = (
        'KEYWORD_IF',
        'KEYWORD_ELSE',
        'KEYWORD_WHILE',
        'KEYWORD_INT',
        'KEYWORD_FLOAT',
        'IDENTIFIER',
        'INTEGER_LITERAL',
        'FLOAT_LITERAL',
        'ASSIGN',
        'PLUS',
        'MINUS',
        'TIMES',
        'DIVIDE',
        'SEMICOLON',
        'LPAREN',
        'RPAREN',
        'LBRACE',
        'RBRACE',
    )

    # 2. DICCIONARIO ASOCIATIVO DE PALABRAS RESERVADAS
    reserved = {
        'if': 'KEYWORD_IF',
        'else': 'KEYWORD_ELSE',
        'while': 'KEYWORD_WHILE',
        'int': 'KEYWORD_INT',
        'float': 'KEYWORD_FLOAT'
    }

    # 3. REGLAS SIMPLES DE CADENA (t_TOKEN)
    t_ASSIGN    = r'='
    t_PLUS      = r'\+'
    t_MINUS     = r'-'
    t_TIMES     = r'\*'
    t_DIVIDE    = r'/'
    t_SEMICOLON = r';'
    t_LPAREN    = r'\('
    t_RPAREN    = r'\)'
    t_LBRACE    = r'\{'
    t_RBRACE    = r'\}'

    # Caracteres ignorados (espacios y tabuladores)
    t_ignore = ' \t'

    # 4. REGLAS BASADAS EN FUNCIONES (t_TOKEN(t))
    def t_FLOAT_LITERAL(self, t):
        r'\d+\.\d+([eE][+-]?\d+)?'
        t.value = float(t.value)
        return t

    def t_INTEGER_LITERAL(self, t):
        r'\d+'
        t.value = int(t.value)
        return t

    def t_IDENTIFIER(self, t):
        r'[a-zA-Z_][a-zA-Z0-9_]*'
        # Disambiguación entre palabras reservadas e identificadores de usuario
        t.type = self.reserved.get(t.value, 'IDENTIFIER')
        return t

    def t_newline(self, t):
        r'\n+'
        t.lexer.lineno += len(t.value)

    def t_error(self, t):
        print(f"⚠️  [Error Léxico] Línea {t.lexer.lineno}, Posición {t.lexpos}: Carácter ilegal '{t.value[0]}'")
        t.lexer.skip(1)

    def build(self, **kwargs):
        """Construye el objeto lexer dinámico mediante PLY."""
        self.lexer = lex.lex(object=self, **kwargs)

    def test(self, data: str):
        """Prueba el escáner sobre una cadena fuente imprimiendo los tokens capturados."""
        self.lexer.input(data)
        print(f"{'Línea':<8} | {'Tipo de Token':<18} | {'Lexema / Valor':<20} | {'Posición Abs.':<12}")
        print("-" * 65)
        for tok in self.lexer:
            print(f"{tok.lineno:<8} | {tok.type:<18} | {str(tok.value):<20} | {tok.lexpos:<12}")

# Práctica Base
sample_code = """
int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

#print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
#py_lexer = PythonLexerPLY()
#py_lexer.build()
#py_lexer.test(sample_code)

Writing lexer_ply.py


In [3]:
from lexer_ply import PythonLexerPLY, sample_code

print("=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===")
py_lexer = PythonLexerPLY()
py_lexer.build()
py_lexer.test(sample_code)

=== INICIANDO PRUEBA DEL ANALIZADOR LÉXICO EN PLY (Python) ===
Línea    | Tipo de Token      | Lexema / Valor       | Posición Abs.
-----------------------------------------------------------------
2        | KEYWORD_INT        | int                  | 1           
2        | IDENTIFIER         | contador             | 5           
2        | ASSIGN             | =                    | 14          
2        | INTEGER_LITERAL    | 10                   | 16          
2        | SEMICOLON          | ;                    | 18          
3        | KEYWORD_FLOAT      | float                | 20          
3        | IDENTIFIER         | promedio             | 26          
3        | ASSIGN             | =                    | 35          
3        | FLOAT_LITERAL      | 95.5                 | 37          
3        | SEMICOLON          | ;                    | 41          
4        | KEYWORD_WHILE      | while                | 43          
4        | LPAREN             | (                    |

### 🛠️ Desafío 1 (PLY - Python)
**Instrucciones:** Escribe a continuación la extensión de la clase `PythonLexerPLY` (o redefine la clase ampliada `PythonLexerPLYExtended`) para incluir:
1. Soporte para **cadenas literales** entrecomilladas (`"..."`) asignando el token `'STRING_LITERAL'`.
2. Soporte para **comentarios de una línea** (`//...`), configurando el lexer para que sean **ignorados en silencio** sin emitir ningún token.

In [4]:
# [ESPACIO PARA EL ALUMNO] - Escribe tu código ampliado de PLY aquí
# Pista: Agrega 'STRING_LITERAL' a tokens y define t_STRING_LITERAL y t_ignore_COMMENT


---  
## ☕ PARTE 2: Analizador Léxico en Java utilizando JFlex

### 2.1 Configuración e Instalación del Entorno

In [5]:
import os

print("=== 1. INSTALANDO JDK EN GOOGLE COLAB ===")
ret = os.system("apt-get update -qq && apt-get install -y openjdk-11-jdk-headless > /dev/null")
print("✅ Entorno Java 11 configurado correctamente.\n" if ret == 0 else "❌ Falló la instalación del JDK.\n")

=== 1. INSTALANDO JDK EN GOOGLE COLAB ===
✅ Entorno Java 11 configurado correctamente.



### 2.2 Creación de Archivos Base de Java y Especificación JFlex

In [6]:
# --- PASO 1: DEFINICIÓN DE LA ENUMERACIÓN DE TOKENS (TokenType.java) ---
tokentype_code = """
public enum TokenType {
    // Palabras Reservadas
    KEYWORD_IF, KEYWORD_ELSE, KEYWORD_WHILE, KEYWORD_INT, KEYWORD_FLOAT,

    // Identificadores y Literales
    IDENTIFIER, INTEGER_LITERAL, FLOAT_LITERAL, STRING_LITERAL,

    // Operadores y Delimitadores
    ASSIGN, PLUS, MINUS, TIMES, DIVIDE, SEMICOLON,
    LPAREN, RPAREN, LBRACE, RBRACE,

    // Control de Errores
    ERROR
}
"""

with open("TokenType.java", "w") as f:
    f.write(tokentype_code)

# --- PASO 2: CLASE CONTENEDORA DE TOKENS CON RASTREO (Token.java) ---
token_code = """
public class Token {
    public TokenType type;
    public Object value;
    public int line;
    public int column;

    public Token(TokenType type, Object value, int line, int column) {
        this.type = type;
        this.value = value;
        this.line = line;
        this.column = column;
    }

    @Override
    public String toString() {
        return String.format("%-8d | %-8d | %-18s | %-20s", line, column, type, value);
    }
}
"""

with open("Token.java", "w") as f:
    f.write(token_code)

# --- PASO 3: ARCHIVO DE ESPECIFICACIÓN JFLEX (scanner.jflex) ---
jflex_spec_code = """
import java.io.*;

%%

/* SECCIÓN 1: OPCIONES Y DIRECTIVAS DE CONFIGURACIÓN */
%class Lexer
%public
%unicode
%line
%column
%type Token

%{
    // Métodos auxiliares para construir objetos Token con coordenadas exactas
    private Token symbol(TokenType type) {
        return new Token(type, yytext(), yyline + 1, yycolumn + 1);
    }

    private Token symbol(TokenType type, Object value) {
        return new Token(type, value, yyline + 1, yycolumn + 1);
    }
%}

/* SECCIÓN 2: MACROS Y DEFINICIONES REGULARES */
DIGITO      = [0-9]
LETRA       = [a-zA-Z_]
ID          = {LETRA}({LETRA}|{DIGITO})*
ENTERO      = {DIGITO}+
FLOTANTE    = {DIGITO}+\\.{DIGITO}+([eE][+-]?{DIGITO}+)?
WS          = [ \\t\\r\\n]+

%%

/* SECCIÓN 3: REGLAS LÉXICAS Y ACCIONES EN JAVA */

/* Ignorar espacios en blanco, tabuladores y saltos de línea */
{WS}            { /* Ignorar */ }

/* Palabras Reservadas (prioridad por ubicación superior a ID) */
"if"            { return symbol(TokenType.KEYWORD_IF); }
"else"          { return symbol(TokenType.KEYWORD_ELSE); }
"while"         { return symbol(TokenType.KEYWORD_WHILE); }
"int"           { return symbol(TokenType.KEYWORD_INT); }
"float"         { return symbol(TokenType.KEYWORD_FLOAT); }

/* Literales con conversión de tipo nativo */
{FLOTANTE}      { return symbol(TokenType.FLOAT_LITERAL, Double.parseDouble(yytext())); }
{ENTERO}        { return symbol(TokenType.INTEGER_LITERAL, Integer.parseInt(yytext())); }

/* Identificadores */
{ID}            { return symbol(TokenType.IDENTIFIER, yytext()); }

/* Operadores y Delimitadores */
"="             { return symbol(TokenType.ASSIGN); }
"+"             { return symbol(TokenType.PLUS); }
"-"             { return symbol(TokenType.MINUS); }
"*"             { return symbol(TokenType.TIMES); }
"/"             { return symbol(TokenType.DIVIDE); }
";"             { return symbol(TokenType.SEMICOLON); }
"("             { return symbol(TokenType.LPAREN); }
")"             { return symbol(TokenType.RPAREN); }
"{"             { return symbol(TokenType.LBRACE); }
"}"             { return symbol(TokenType.RBRACE); }

/* Captura de Errores Léxicos (Modo Pánico) */
.               {
                    System.err.printf("⚠️  [Error Léxico] Línea %d, Columna %d: Carácter ilegal '%s'\\n",
                                      yyline + 1, yycolumn + 1, yytext());
                    return symbol(TokenType.ERROR, yytext());
                }
"""

with open("scanner.jflex", "w") as f:
    f.write(jflex_spec_code)

# --- PASO 4: CLASE PRINCIPAL DE PRUEBA (Main.java) ---
main_code = """
import java.io.*;

public class Main {
    public static void main(String[] args) {
        if (args.length == 0) {
            System.out.println("Por favor ingrese la ruta del archivo de prueba.");
            return;
        }

        try {
            FileReader reader = new FileReader(args[0]);
            Lexer scanner = new Lexer(reader);

            System.out.println("=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===");
            System.out.println(String.format("%-8s | %-8s | %-18s | %-20s", "Línea", "Columna", "Tipo de Token", "Lexema / Valor"));
            System.out.println("-".repeat(65));

            Token token;
            while ((token = scanner.yylex()) != null) {
                System.out.println(token);
            }
            System.out.println("\\n=== ESCANEO FINALIZADO EXITOSAMENTE ===");

        } catch (IOException e) {
            System.err.println("Error al leer el archivo: " + e.getMessage());
        }
    }
}
"""

with open("Main.java", "w") as f:
    f.write(main_code)

# --- PASO 5: ARCHIVO DE FUENTE DE PRUEBA (prueba.txt) ---
sample_code = """int contador = 10;
float promedio = 95.5;
while (contador) {
    promedio = promedio + 1.2e2;
    if (promedio) {
        @
    }
}
"""

with open("prueba.txt", "w") as f:
    f.write(sample_code)

print("✅ Archivos Java y especificacion scanner.jflex creados exitosamente.")

✅ Archivos Java y especificacion scanner.jflex creados exitosamente.


### 2.3 Compilación y Ejecución en Colab

In [7]:
print("=== DESCARGANDO JFLEX (JAR con dependencias) ===")
!rm -rf jflex-1.9.1 jflex.tar.gz
!wget -q https://github.com/jflex-de/jflex/releases/download/v1.9.1/jflex-1.9.1.tar.gz -O jflex.tar.gz
!tar -xzf jflex.tar.gz

print("\n=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===")
!java -Xlog:disable -jar jflex-1.9.1/lib/jflex-full-1.9.1.jar scanner.jflex
!javac -J-Xlog:disable TokenType.java Token.java Lexer.java Main.java

print("\n=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===")
!java -Xlog:disable Main prueba.txt

=== DESCARGANDO JFLEX (JAR con dependencias) ===

=== GENERANDO Y COMPILANDO ARCHIVOS JAVA CON JFLEX ===
Reading "scanner.jflex"
Constructing NFA : 80 states in NFA
Converting NFA to DFA : 
........................................
42 states before minimization, 38 states in minimized DFA
Writing code to "Lexer.java"

=== SALIDA DEL ESCÁNER PROCESANDO 'prueba.txt' ===
=== ANALIZADOR LÉXICO GENERADO POR JFLEX (Java) ===
Línea    | Columna  | Tipo de Token      | Lexema / Valor      
-----------------------------------------------------------------
1        | 1        | KEYWORD_INT        | int                 
1        | 5        | IDENTIFIER         | contador            
1        | 14       | ASSIGN             | =                   
1        | 16       | INTEGER_LITERAL    | 10                  
1        | 18       | SEMICOLON          | ;                   
2        | 1        | KEYWORD_FLOAT      | float               
2        | 7        | IDENTIFIER         | promedio            


### 🛠️ Desafío 2 (JFlex - Java)
**Instrucciones:** Modifica la especificación `scanner.jflex` arriba o vuelve a escribir el archivo desde una celda agregando:
1. La definición de la regla de **cadenas literales entrecomilladas**: `\"[^\"]*\" { return symbol(TokenType.STRING_LITERAL, yytext()); }`
2. La regla para **comentarios de una línea**: `"//".* { /* Ignorar */ }`
Recompila con `!jflex scanner.jflex && javac *.java` y ejecuta sobre un archivo de prueba ampliado.

---  
## 📝 BITÁCORA Y REPORTE DE LABORATORIO

**Nombre del Estudiante:** Emanuel Izaguirre Cortes__________________________________  
**Matrícula / Código:** a2223330168_____________________________________  
**Fecha:** ___________27 de nov 2026 __________  

---
### Sección 1: Verificación de Coincidencia y Resolución de Ambigüedades
1. **Mecanismo de la regla más larga (*Longest Match*):**  
   Al analizar la subcadena `1.2e2`, explica paso a paso por qué el autómata generado no la separa como el entero `1`, el operador `.` y el identificador/entero `2e2`, sino como un único token `FLOAT_LITERAL`.  
   *Respuesta del alumno:*
   > El principio de **Coincidencia más Larga (Longest Match)** determina que un analizador léxico siempre intentará consumir la mayor cantidad de caracteres posibles que coincidan con cualquier patrón válido antes de delimitar un token. Al evaluar `1.2e2`, el analizador tiene dos caminos: capturar por separado `1` (entero), `.` (punto) y `2e2` (identificador), o agrupar todo como un único número de coma flotante en notación científica. Dado que toda la secuencia `1.2e2` (caracteres del 1 al 5) coincide en su totalidad con el patrón de la expresión regular definida para `FLOAT_LITERAL` (`\d+\.\d+([eE][+-]?\d+)?`), el autómata prioriza esta coincidencia única de 5 caracteres sobre las tres conincidencias individuales más cortas.

2. **Disambiguación de Palabras Reservadas vs. Identificadores:**  
   * En **JFlex**, ¿por qué la posición (física) de las reglas de `"if"`, `"while"`, etc., por encima de la regla `{ID}` garantiza que no sean clasificadas como variables?  
   * En **PLY**, ¿por qué se utiliza el método `reserved.get(t.value, 'IDENTIFIER')` dentro de la función `t_IDENTIFIER` en lugar de crear una regla independiente `t_KEYWORD_IF = r'if'`?  
   *Respuesta del alumno:*
   > * **En JFlex:** El generador de JFlex maneja los conflictos de prioridad mediante el orden secuencial de aparición física de las reglas en el archivo de especificación. Si una misma cadena (ej. `"if"`) coincide con múltiples patrones (en este caso el lexema literal `"if"` y el patrón general `{ID}`), JFlex prioriza la regla definida primero (arriba). Por tanto, declararlas arriba garantiza que se cataloguen como palabras clave antes de que puedan capturarse de manera genérica.
   > * **En PLY:** En PLY, todas las reglas de tipo `t_NOMBRE = r'...'` se combinan internamente en una única macro-expresión regular gigante donde las funciones de token (`t_` basadas en métodos) se ordenan según su aparición, pero declarar reglas independientes para cada palabra clave puede generar ambigüedades innecesarias o sobrecarga en la búsqueda. La forma idiomática y de mejor rendimiento es capturar todo lexema candidato a palabra clave como `IDENTIFIER` mediante expresiones regulares básicas, y luego realizar una búsqueda directa en un diccionario asociativo dinámico (`reserved.get(t.value, 'IDENTIFIER')`) antes de clasificar el token definitivo.

---
### Sección 2: Conversión y Tipado de Atributos Léxicos
1. **Conversión de Tipos:**  
   En la regla `t_FLOAT_LITERAL` de PLY se ejecuta `t.value = float(t.value)`, y en JFlex se pasa `Double.parseDouble(yytext())`. ¿Por qué es fundamental que esta conversión ocurra dentro del analizador léxico antes de entregar el token al analizador sintáctico/semántico?  
   *Respuesta del alumno:*
   > Realizar esta conversión dentro del analizador léxico garantiza que las fases de análisis sintáctico y semántico reciban de inmediato un valor con un tipo de dato nativo robusto (`float`, `int`, `Double`), en lugar de una simple secuencia de caracteres crudos (Strings). Esto reduce la complejidad de las rutinas sintácticas superiores, permitiendo operaciones semánticas inmediatas (evaluación constante de expresiones, validación de tipos) y previniendo errores de casteo dispersos a lo largo del compilador.

---
### Sección 3: Análisis Estructural y Manejo de Posiciones
1. **Comparación de Rastreos de Coordenadas:**  
   Explica la diferencia técnica entre cómo rastrea PLY la ubicación de los tokens (utilizando `t.lexer.lineno` y `t.lexpos`) y cómo lo realiza JFlex mediante las directivas `%line` y `%column` con las variables internas `yyline` y `yycolumn`.  
   *Respuesta del alumno:*
   > * **En PLY:** El desarrollador es responsable de actualizar explícitamente el contador de líneas (`t.lexer.lineno += len(t.value)`) al encontrar un salto de línea (`\n`). Además, PLY solo provee la posición absoluta del inicio del token en el archivo original mediante `t.lexpos`, requiriendo de algoritmos auxiliares si se desea obtener la columna exacta.
   > * **En JFlex:** Al incluir las directivas `%line` y `%column`, JFlex genera autómatas que gestionan de forma nativa y automática el incremento del contador de filas (`yyline`) y columnas (`yycolumn`) según cada carácter procesado, independientemente de la lógica de cada token individual.

---
### Sección 4: Extensión de Reglas (Evidencia del Desafío)
Pega las líneas de código o expresiones regulares que agregaste en PLY y JFlex para dar soporte a cadenas literales y comentarios de una línea:

**Código/Reglas agregadas en PLY (Python):**

```python
# 1. Definición del token en la tupla tokens
tokens = (
    # ... otros tokens ...
    'STRING_LITERAL',
)

# 2. Regla para omitir comentarios de una línea (en silencio)
def t_ignore_COMMENT(self, t):
    r'//.*'
    pass

# 3. Regla para capturar cadenas literales
def t_STRING_LITERAL(self, t):
    r'"[^"]*"'
    t.value = t.value[1:-1] # Elimina las comillas exteriores
    return t
```

**Reglas agregadas en scanner.jflex (Java):**

```jflex
/* 1. Definición del token en TokenType.java */
STRING_LITERAL

/* 2. Regla para ignorar comentarios en scanner.jflex */
"//".*          { /* Ignorar */ }

/* 3. Regla para capturar cadenas de texto literales */
\"[^\"]*\"      { return symbol(TokenType.STRING_LITERAL, yytext()); }
```

---
### Sección 5: Cuadro Comparativo Arquitectónico

| Criterio de Comparación | PLY (Python) | JFlex (Java) |
| :--- | :--- | :--- |
| **Naturaleza del Generador** | Reflexivo / Dinámico (Interpretado en ejecución) | Generador Estático / Compilado |
| **Definición de Patrones** | Expresiones regulares escritas en los docstrings | Sintaxis nativa en archivo de especificación `.jflex` |
| **Manejo de Palabras Reservadas** | Filtrado manual con diccionarios y `reserved.get` | Reglas literales físicas ordenadas con precedencia superior |
| **Rastreo de Línea y Columna** | Manual en el lexer (`t.lexer.lineno`) y `lexpos` | Automático mediante directivas `%line` y `%column` |
| **Recuperación de Errores** | Método `t_error` para manejar saltos (`t.lexer.skip`) | Regla comodín `.` con lógica Java personalizada |

---  
**Conclusión General del Experimento:**  
> El uso de generadores automatizados de analizadores léxicos como PLY y JFlex optimiza drásticamente el tiempo de desarrollo en la construcción de compiladores en comparación con la escritura manual de autómatas de estado. JFlex destaca por su robustez estática y rendimiento en tiempo de ejecución, ideal para compiladores industriales en Java. Por otra parte, PLY aprovecha la reflexividad de Python permitiendo un prototipado extremadamente ágil y con menos archivos intermedios.